Exponential Kernel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from scipy.ndimage import gaussian_filter1d
from data import EventData
from StatisticEstimators import *
from MHP import MHP

# Ground-truth parameters 
alpha = np.array([[0.30, 0.10], 
                  [0.20, 0.25]])
omega = 3.0
mu    = np.array([0.3, 0.2])
D     = 2

Lambda_true = np.linalg.solve(np.eye(D) - alpha, mu)

# Simulate 
horizon = 50_000.0
hp  = MHP(Phi=alpha, mu=mu, omega=omega)
hp.check_stability()
raw = hp.generate(horizon=horizon, seed=0)
print(f"Simulated {len(raw):,} events  (T={horizon:.0f})")

events = EventData(
    times  = raw[:, 0],
    types  = raw[:, 1].astype(int),
    marks  = np.ones(len(raw)),   # dummy  unused cause we can't find some in this simulation only for like csv
    horizon= horizon,
)

# Estimate statistics 

Lambda_hat       = estimate_Lambda(events)
lag_max, h       = 1.0, 0.15
G_hat, t_grid, _ = estimate_G(events, Lambda_hat,
                               lag_max=lag_max, h=h, n_lin=8, n_log=30)
# Spline-smooth in log-time 
G_smooth = smooth_G_spline(G_hat, t_grid, smooth_factor=None)

print(f"True  Î» : {np.round(Lambda_true, 4)}")
print(f"Est.  Î» : {np.round(Lambda_hat,  4)}")
print(f"Error   : {np.round(np.abs(Lambda_hat - Lambda_true), 5)}")

#true G 
A   = omega * (alpha - np.eye(D))
B   = omega * alpha @ np.diag(Lambda_true)
Gss = -np.linalg.inv(A) @ B
C   = alpha * omega

G_true = np.zeros((len(t_grid), D, D))
for k, t in enumerate(t_grid):
    G_true[k] = expm(A * t) @ C + Gss

# Plot 
CT, CE, CS = "#1C3F6E", "#D95F02", "#2CA02C"
label = {(0,0): "self  1â†’1", (0,1): "cross 2â†’1",
         (1,0): "cross 1â†’2", (1,1): "self  2â†’2"}

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
fig.suptitle(
    rf"$\hat G$ (StatisticEstimators.py + EventData)  vs  True $G$"
    f"\nT={horizon:.0f},  N={len(raw):,},  "
    r"$\alpha$=[[0.40,0.20],[0.30,0.35]],  $\omega$=5",
    fontsize=13, fontweight="bold"
)

for i in range(D):
    for j in range(D):
        ax = axes[i, j]
        tv = G_true[:, i, j]
        ev = G_hat[i, j, :]
        sv = G_smooth[i, j, :]

        rl2_raw = (np.sqrt(np.trapezoid((tv-ev)**2, t_grid))
                   / (np.sqrt(np.trapezoid(tv**2, t_grid)) + 1e-12)) * 100
        rl2_smo = (np.sqrt(np.trapezoid((tv-sv)**2, t_grid))
                   / (np.sqrt(np.trapezoid(tv**2, t_grid)) + 1e-12)) * 100

        ax.plot(t_grid, ev, color=CE, lw=0.9, alpha=0.5,
                label=rf"$\hat G^{{{i+1}{j+1}}}$ raw  (L2={rl2_raw:.0f}%)")
        ax.plot(t_grid, sv, color=CS, lw=2.0, ls="-.",
                label=rf"$\hat G^{{{i+1}{j+1}}}$ smoothed  (L2={rl2_smo:.0f}%)")
        ax.plot(t_grid, tv, color=CT, lw=2.5,
                label=rf"True $G^{{{i+1}{j+1}}}$")
        ax.axhline(0, color="gray", lw=0.7, ls=":")

        ax.set_title(rf"$G^{{{i+1}{j+1}}}(t)$ â€” {label[(i,j)]}", fontsize=11)
        ax.set_ylabel(rf"$G^{{{i+1}{j+1}}}(t)$")
        if i == 1:
            ax.set_xlabel("lag $t$")
        ax.legend(fontsize=8.5, loc="upper right")
        ax.grid(ls="--", alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import copy
import numpy as np
import matplotlib.pyplot as plt
from models import KernelRowNet
from StatisticEstimators import build_H
from loss import temporal_weights
device = torch.device('cpu')

# Hyperparameters
HIDDEN      = 64       # neurons per hidden layer
LAYERS      = 1        # DGM depth (number of hidden layers)
LR0         = 1e-3     # learning rate
Q           = 250      # quadrature points (for integral approximation)
BATCH_SIZE  = 8        # collocation mini-batch size per gradient step
TRAIN_SIZE  = 1024     # total collocation training points
VAL_SIZE    = 128      # collocation validation points
EPOCHS      = 500     # training epochs
EPS_W       = 5.0      # epsilon for temporal_weights 
PRINT_EVERY = 100

t_min = t_grid[0]
T     = lag_max

# Fixed quadrature grid (for integral approximation)
e1 = np.logspace(np.log10(t_min), np.log10(0.05),  80 + 1)
e2 = np.logspace(np.log10(0.05),  np.log10(0.3),   100 + 1)
e3 = np.logspace(np.log10(0.3),   np.log10(T),      70 + 1)
edges = np.unique(np.concatenate([e1, e2[1:], e3[1:]]))
t_np  = 0.5 * (edges[:-1] + edges[1:])
w_np  = np.diff(edges)
t_th  = torch.tensor(t_np, dtype=torch.float32)
w_th  = torch.tensor(w_np, dtype=torch.float32)

# Random collocation grids 
rng        = np.random.default_rng(42)
t_train_np = np.sort(rng.uniform(t_min, T, TRAIN_SIZE))
t_val_np   = np.sort(rng.uniform(t_min, T, VAL_SIZE))

# Precompute H for collocation grids
H_func = build_H(G_smooth, t_grid, Lambda_hat)

def compute_H_coll(t_coll, t_quad, H_func, D):
    N, Qq = len(t_coll), len(t_quad)
    diff  = t_coll[:, None] - t_quad[None, :]   # (N, Q)
    H     = np.zeros((N, Qq, D, D))
    for k in range(D):
        for j in range(D):
            H[:, :, k, j] = H_func(k, j, diff.ravel()).reshape(N, Qq)
    return torch.tensor(H, dtype=torch.float32)

print('Precomputing H for training collocation points ')
H_train_th = compute_H_coll(t_train_np, t_np, H_func, D)   # (TRAIN_SIZE, Q, D, D)
print('Precomputing H for validation collocation points ')
H_val_th   = compute_H_coll(t_val_np,   t_np, H_func, D)   # (VAL_SIZE,   Q, D, D)

# Targets: G_smooth interpolated at collocation points
def interp_G_target(t_coll):
    G_interp = np.zeros((D, D, len(t_coll)))
    for i in range(D):
        for j in range(D):
            G_interp[i, j] = np.interp(t_coll, t_grid, G_smooth[i, j])
    return torch.tensor(G_interp, dtype=torch.float32)

G_train_th = interp_G_target(t_train_np)   # (D, D, TRAIN_SIZE)
G_val_th   = interp_G_target(t_val_np)     # (D, D, VAL_SIZE)

# Log-time inputs (training points are pre-sorted for temporal_weights)
log_t_train = torch.log10(torch.tensor(t_train_np, dtype=torch.float32).clamp_min(1e-8))
x_train     = torch.ones_like(log_t_train)
log_t_val   = torch.log10(torch.tensor(t_val_np,   dtype=torch.float32).clamp_min(1e-8))
x_val       = torch.ones_like(log_t_val)
log_t_quad  = torch.log10(t_th.clamp_min(1e-8))
x_quad      = torch.ones_like(log_t_quad)

# Training loop
def train_row_dgm(row_i):
    model = KernelRowNet(input_dim=2, hidden_dim=HIDDEN, output_dim=D, n_layers=LAYERS).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=LR0, weight_decay=1e-6)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

    target_train = G_train_th[row_i].T   # (TRAIN_SIZE, D)
    target_val   = G_val_th[row_i].T     # (VAL_SIZE, D)

    best_val, best_epoch, best_state = float('inf'), -1, None
    patience, no_improve = 120, 0
    steps_per_epoch = TRAIN_SIZE // BATCH_SIZE

    for ep in range(EPOCHS):
        
        model.eval()
        with torch.no_grad():
            phi_all    = model(log_t_train, x_train)                             # (TRAIN_SIZE, D)
            phi_q_ng   = model(log_t_quad, x_quad)                               # (Q, D)
            int_all    = torch.einsum('qk,nqkj->nj', phi_q_ng * w_th[:, None], H_train_th)  # (TRAIN_SIZE, D)
            res_all    = phi_all + int_all - target_train                        # (TRAIN_SIZE, D)
            w_all      = temporal_weights(res_all, eps=EPS_W)                   # (TRAIN_SIZE, D) 

        model.train()
        perm       = torch.randperm(TRAIN_SIZE)
        epoch_loss = 0.0

        for step in range(steps_per_epoch):
            idx   = perm[step * BATCH_SIZE : (step + 1) * BATCH_SIZE]
            H_b   = H_train_th[idx]          # (BATCH_SIZE, Q, D, D)
            tgt_b = target_train[idx]        # (BATCH_SIZE, D)
            w_b   = w_all[idx]               # (BATCH_SIZE, D)  fixed, Eq.(22)

            phi_b    = model(log_t_train[idx], x_train[idx])        # (BATCH_SIZE, D)
            phi_quad = model(log_t_quad, x_quad)                     # (Q, D)
            integral = torch.einsum('qk,nqkj->nj', phi_quad * w_th[:, None], H_b)  # (BATCH_SIZE, D)
            residual = phi_b + integral - tgt_b                      # eps_n^ij 

            # loss with weights 
            loss = (w_b * residual.square()).mean()

            opt.zero_grad()
            loss.backward()
            opt.step()
            epoch_loss += loss.item()

        sched.step()

        model.eval()
        with torch.no_grad():
            phi_v  = model(log_t_val, x_val)
            phi_qv = model(log_t_quad, x_quad)
            int_v  = torch.einsum('qk,nqkj->nj', phi_qv * w_th[:, None], H_val_th)
            val_loss = (phi_v + int_v - target_val).square().mean().item()  # Eq.(25)

        if val_loss < best_val:
            best_val, best_epoch = val_loss, ep
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if ep % PRINT_EVERY == 0 or ep == EPOCHS - 1:
            print(f'  [row {row_i}] ep {ep:04d}  lr={opt.param_groups[0]["lr"]:.3e}  '
                  f'train={epoch_loss/steps_per_epoch:.4e}  val={val_loss:.4e}  '
                  f'best={best_val:.4e}@ep{best_epoch}')


    if best_state is not None:
        model.load_state_dict(best_state)
    print(f'  -> loaded best checkpoint (ep {best_epoch}, val={best_val:.4e})')
    return model


trained_models = []
for row_i in range(D):
    print(f"\n{'='*60}\nTraining row {row_i}\n{'='*60}")
    trained_models.append(train_row_dgm(row_i))


# Predict on fine grid
t_fine   = np.logspace(np.log10(t_np[0]), np.log10(t_np[-1]), 300)
log_fine = torch.log10(torch.tensor(t_fine, dtype=torch.float32).clamp_min(1e-8))
x_fine   = torch.ones_like(log_fine)

phi_pred      = np.zeros((D, D, len(t_fine)))
phi_true_fine = np.zeros_like(phi_pred)

for i in range(D):
    trained_models[i].eval()
    with torch.no_grad():
        phi_pred[i] = trained_models[i](log_fine, x_fine).numpy().T

for i in range(D):
    for j in range(D):
        phi_true_fine[i, j] = alpha[i, j] * omega * np.exp(-omega * t_fine)

# Plot
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
fig.suptitle(
    r'NN kernel $\hat\phi$ vs True $\phi$  --  DGM + paper loss Eq.(26)'
    f'\n(neurons={HIDDEN}, depth={LAYERS}, Q={Q}, batch={BATCH_SIZE}, '
    f'train={TRAIN_SIZE}, val={VAL_SIZE}, epochs={EPOCHS})',
    fontsize=13, fontweight='bold')
lbl = {(0,0): 'self 1->1', (0,1): 'cross 2->1',
       (1,0): 'cross 1->2', (1,1): 'self 2->2'}

for i in range(D):
    for j in range(D):
        ax  = axes[i, j]
        pt  = phi_true_fine[i, j]
        pp  = phi_pred[i, j]
        rl2 = (np.sqrt(np.trapezoid((pt - pp)**2, t_fine))
               / (np.sqrt(np.trapezoid(pt**2, t_fine)) + 1e-12)) * 100
        ax.plot(t_fine, pt, color='#1C3F6E', lw=2.5, label=f'True  $\phi_{i+1}{j+1}$')
        ax.plot(t_fine, pp, color='#D95F02', lw=2.0, ls='--',
                label=f'NN  $\phi_{i+1}{j+1}  (L2={rl2:.0f}%)')
        ax.axhline(0, color='gray', lw=0.7, ls=':')
        ax.set_xscale('log')
        ax.set_title(f'$phi_{i+1}{j+1}(t)$ -- {lbl[(i,j)]}', fontsize=11)
        ax.set_ylabel(f'$phi_{i+1}{j+1}(t)$')
        if i == 1:
            ax.set_xlabel('lag  t')
        ax.legend(fontsize=9)
        ax.grid(ls='--', alpha=0.3)
        ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
from WH import WienerHopf

# Wiener-Hopf estimation on the same grid
wh_model = WienerHopf(n_fft=65536, tikhonov=1e-4).fit(G_smooth, t_grid)

phi_wh = wh_model.predict(t_fine)   # (D, D, len(t_fine))

K_wh = wh_model.kernel_norms()

#comparison plot
lbl = {(0,0): "self 1->1", (0,1): "cross 2->1",
       (1,0): "cross 1->2", (1,1): "self 2->2"}

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
fig.suptitle(
    r"Kernel comparison — True $\phi$ vs Neural Hawkes vs Wiener-Hopf"
    f"\n(T={horizon:.0f}, N={len(raw):,}, alpha=[[0.30,0.10],[0.20,0.25]]",
    fontsize=13, fontweight="bold",
)

C_TRUE = "#1C3F6E"
C_NN   = "#D95F02"
C_WH   = "#2CA02C"

for i in range(D):
    for j in range(D):
        ax = axes[i, j]
        pt = phi_true_fine[i, j]
        pp = phi_pred[i, j]
        pw = phi_wh[i, j]

        rl2_nn = (np.sqrt(np.trapezoid((pt - pp)**2, t_fine))
                  / (np.sqrt(np.trapezoid(pt**2, t_fine)) + 1e-12)) * 100
        rl2_wh = (np.sqrt(np.trapezoid((pt - pw)**2, t_fine))
                  / (np.sqrt(np.trapezoid(pt**2, t_fine)) + 1e-12)) * 100

        ax.plot(t_fine, pt, color=C_TRUE, lw=2.5,
                label=rf"True $\phi^{{{i+1}{j+1}}}$")
        ax.plot(t_fine, pp, color=C_NN, lw=2.0, ls="--",
                label=rf"NN  (rL2={rl2_nn:.0f}%)")
        ax.plot(t_fine, pw, color=C_WH, lw=2.0, ls="-.",
                label=rf"WH  (rL2={rl2_wh:.0f}%)")

        ax.axhline(0, color="gray", lw=0.7, ls=":")
        ax.set_xscale("log")
        ax.set_title(rf"$\phi^{{{i+1}{j+1}}}(t)$ — {lbl[(i,j)]}", fontsize=11)
        ax.set_ylabel(rf"$\phi^{{{i+1}{j+1}}}(t)$")
        if i == 1:
            ax.set_xlabel("lag  $")
        ax.legend(fontsize=9, loc="upper right")
        ax.grid(ls="--", alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()
